In [2]:
import gc
import inspect
import json
import os
from dataclasses import dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


HF_ENDPOINT: https://hf-mirror.com
HF_HUB_DISABLE_XET: 1


In [ ]:
# Cell 2: DPO 配置，承接 E1_sft 的 9B SFT adapter
DATA_DIR = Path("/home/yangdejin/nlpcc/nlpcc_task2/data")
OUT_DIR = Path("/home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/dpo")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen3.5-9B"
SFT_ADAPTER_DIR = Path("/home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/sft")

TRAIN_FILE = DATA_DIR / "track2" / "dpo_train.jsonl"
DEV_FILE = DATA_DIR / "track2" / "dpo_dev.jsonl"

MAX_LENGTH = 768
MAX_PROMPT_LENGTH = 576

BETA = 0.1
LR = 3e-6
EPOCHS = 1
TRAIN_BS = 1
EVAL_BS = 1
GRAD_ACCUM = 16
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0

BF16 = True
FP16 = False
LOAD_IN_4BIT = True
USE_SFT_ADAPTER = True
PRECOMPUTE_REF_LOGPS = True
REFERENCE_BS = 1

assert USE_SFT_ADAPTER, "DPO should start from the E1 SFT adapter."
print("SFT adapter:", SFT_ADAPTER_DIR)


SFT adapter: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/sft


In [4]:
# Cell 3: 数据读取

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


train_rows = read_jsonl(TRAIN_FILE)
dev_rows = read_jsonl(DEV_FILE)

print("train:", len(train_rows))
print("dev  :", len(dev_rows))
print("\nPrompt sample:\n", train_rows[0]["prompt"][:700])
print("\nChosen:\n", train_rows[0]["chosen"])
print("\nRejected:\n", train_rows[0]["rejected"])


train: 3520
dev  : 514

Prompt sample:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:


Chosen:
 I would suggest alternative solutions diplomatically, ensuring my input is heard without challenging the manager openly, prioritizing team harmony and respect for hierarchy.

Rejected:
 I would publicly challenge the manager's decision, emphasizing my personal principles over maintaining workplace relationships, even if it risks disapproval or tension.


In [5]:
# Cell 4: tokenizer
tokenizer_source = str(SFT_ADAPTER_DIR) if SFT_ADAPTER_DIR.exists() else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

tokenizer.padding_side = "right"

print("tokenizer source:", tokenizer_source)
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)


tokenizer source: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/sft
pad token: <|endoftext|> 248044


In [6]:
# Cell 6: 模型加载函数

def model_init_kwargs():
    model_kwargs = {"trust_remote_code": True}
    if LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig

        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = "auto"
    elif BF16:
        model_kwargs["torch_dtype"] = torch.bfloat16
    elif FP16:
        model_kwargs["torch_dtype"] = torch.float16
    return model_kwargs


def load_base_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_init_kwargs())
    model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    return model


def load_sft_model(is_trainable):
    from peft import PeftModel, prepare_model_for_kbit_training

    model = load_base_model()
    if LOAD_IN_4BIT and is_trainable:
        model = prepare_model_for_kbit_training(model)
    model = PeftModel.from_pretrained(model, str(SFT_ADAPTER_DIR), is_trainable=is_trainable)
    model.config.use_cache = False
    if is_trainable:
        model.gradient_checkpointing_enable()
        if hasattr(model, "print_trainable_parameters"):
            model.print_trainable_parameters()
    else:
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
    if not LOAD_IN_4BIT:
        model = model.cuda() if torch.cuda.is_available() else model
    return model


In [7]:
# Cell 7: DPO Dataset 与 Collator
class DPODataset(Dataset):
    def __init__(self, rows, tokenizer, max_length=1024, max_prompt_length=768):
        self.examples = []
        eos = tokenizer.eos_token or ""
        for row in rows:
            chosen = self.encode_pair(row["prompt"], row["chosen"], tokenizer, eos, max_length, max_prompt_length)
            rejected = self.encode_pair(row["prompt"], row["rejected"], tokenizer, eos, max_length, max_prompt_length)
            self.examples.append({"chosen": chosen, "rejected": rejected})

    def encode_pair(self, prompt, response, tokenizer, eos, max_length, max_prompt_length):
        prompt_ids = tokenizer(
            prompt,
            add_special_tokens=True,
            truncation=True,
            max_length=max_prompt_length,
        )["input_ids"]
        response_ids = tokenizer(
            response + eos,
            add_special_tokens=False,
            truncation=True,
            max_length=max(1, max_length - len(prompt_ids)),
        )["input_ids"]
        input_ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids
        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


@dataclass
class DPOCollator:
    pad_token_id: int

    def pad(self, features, key):
        max_len = max(len(x[key]) for x in features)
        pad_value = -100 if key == "labels" else self.pad_token_id
        if key == "attention_mask":
            pad_value = 0
        return torch.tensor(
            [x[key] + [pad_value] * (max_len - len(x[key])) for x in features],
            dtype=torch.long,
        )

    def __call__(self, features):
        chosen = [x["chosen"] for x in features]
        rejected = [x["rejected"] for x in features]
        batch = {
            "chosen_input_ids": self.pad(chosen, "input_ids"),
            "chosen_attention_mask": self.pad(chosen, "attention_mask"),
            "chosen_labels": self.pad(chosen, "labels"),
            "rejected_input_ids": self.pad(rejected, "input_ids"),
            "rejected_attention_mask": self.pad(rejected, "attention_mask"),
            "rejected_labels": self.pad(rejected, "labels"),
        }
        if "ref_chosen_logp" in features[0]:
            batch["ref_chosen_logp"] = torch.tensor([x["ref_chosen_logp"] for x in features], dtype=torch.float32)
            batch["ref_rejected_logp"] = torch.tensor([x["ref_rejected_logp"] for x in features], dtype=torch.float32)
        return batch


train_dataset = DPODataset(train_rows, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH)
dev_dataset = DPODataset(dev_rows, tokenizer, MAX_LENGTH, MAX_PROMPT_LENGTH)
collator = DPOCollator(tokenizer.pad_token_id)

print("train dataset:", len(train_dataset))
print("dev dataset  :", len(dev_dataset))
print("first chosen length:", len(train_dataset[0]["chosen"]["input_ids"]))


train dataset: 3520
dev dataset  : 514
first chosen length: 134


In [8]:
# Cell 8: DPO loss 与 reference log-prob 预计算

def sequence_logps(model, input_ids, attention_mask, labels):
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :]
    labels = labels[:, 1:].clone()
    loss_mask = labels != -100
    labels = labels.masked_fill(labels == -100, 0)
    token_logps = torch.gather(logits.log_softmax(dim=-1), dim=2, index=labels.unsqueeze(2)).squeeze(2)
    return (token_logps * loss_mask).sum(dim=1)


def move_batch_to_device(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}


def attach_reference_logps(ref_model, dataset, collator, batch_size=1):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)
    ref_chosen, ref_rejected = [], []
    device = next(ref_model.parameters()).device
    ref_model.eval()

    for batch in loader:
        batch = move_batch_to_device(batch, device)
        with torch.no_grad():
            chosen = sequence_logps(
                ref_model,
                batch["chosen_input_ids"],
                batch["chosen_attention_mask"],
                batch["chosen_labels"],
            )
            rejected = sequence_logps(
                ref_model,
                batch["rejected_input_ids"],
                batch["rejected_attention_mask"],
                batch["rejected_labels"],
            )
        ref_chosen.extend(chosen.float().cpu().tolist())
        ref_rejected.extend(rejected.float().cpu().tolist())

    for example, chosen_logp, rejected_logp in zip(dataset.examples, ref_chosen, ref_rejected):
        example["ref_chosen_logp"] = chosen_logp
        example["ref_rejected_logp"] = rejected_logp


class DPOTrainer(Trainer):
    def __init__(self, *args, ref_model=None, beta=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.ref_model = ref_model
        self.beta = beta

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        pi_chosen = sequence_logps(
            model,
            inputs["chosen_input_ids"],
            inputs["chosen_attention_mask"],
            inputs["chosen_labels"],
        )
        pi_rejected = sequence_logps(
            model,
            inputs["rejected_input_ids"],
            inputs["rejected_attention_mask"],
            inputs["rejected_labels"],
        )

        if "ref_chosen_logp" in inputs:
            ref_chosen = inputs["ref_chosen_logp"].to(pi_chosen.device)
            ref_rejected = inputs["ref_rejected_logp"].to(pi_rejected.device)
        else:
            with torch.no_grad():
                ref_chosen = sequence_logps(
                    self.ref_model,
                    inputs["chosen_input_ids"],
                    inputs["chosen_attention_mask"],
                    inputs["chosen_labels"],
                )
                ref_rejected = sequence_logps(
                    self.ref_model,
                    inputs["rejected_input_ids"],
                    inputs["rejected_attention_mask"],
                    inputs["rejected_labels"],
                )

        pi_logratios = pi_chosen - pi_rejected
        ref_logratios = ref_chosen - ref_rejected
        logits = self.beta * (pi_logratios - ref_logratios)
        loss = -F.logsigmoid(logits).mean()

        if return_outputs:
            return loss, {"chosen_logps": pi_chosen.detach(), "rejected_logps": pi_rejected.detach()}
        return loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            loss = self.compute_loss(model, inputs)
        return loss.detach(), None, None


In [9]:
# Cell 9: 先预计算 reference log-probs，再只保留 policy model 训练
if PRECOMPUTE_REF_LOGPS:
    print("Loading frozen SFT reference model...")
    ref_model = load_sft_model(is_trainable=False)
    print("Precomputing train reference log-probs...")
    attach_reference_logps(ref_model, train_dataset, collator, batch_size=REFERENCE_BS)
    print("Precomputing dev reference log-probs...")
    attach_reference_logps(ref_model, dev_dataset, collator, batch_size=REFERENCE_BS)
    del ref_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    ref_model_for_trainer = None
else:
    ref_model_for_trainer = load_sft_model(is_trainable=False)

print("Loading trainable SFT policy model...")
policy_model = load_sft_model(is_trainable=True)

training_kwargs = dict(
    output_dir=str(OUT_DIR),
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=10,
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    bf16=BF16,
    fp16=FP16,
    optim="paged_adamw_8bit" if LOAD_IN_4BIT else "adamw_torch",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
)
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs["eval_strategy"] = "steps"
else:
    training_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**training_kwargs)

trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model_for_trainer,
    beta=BETA,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=collator,
)

trainer


Loading frozen SFT reference model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Precomputing train reference log-probs...
Precomputing dev reference log-probs...
Loading trainable SFT policy model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 29,097,984 || all params: 8,980,910,592 || trainable%: 0.3240


In [10]:
trainer.train()
trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))


Step,Training Loss,Validation Loss
100,0.022106,0.001265
200,0.001568,0.000522
220,0.000220,0.000523


/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/home/yangdejin/miniconda3/envs/nlpcc/lib/python3.12/site-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


('/home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/dpo/tokenizer_config.json',
 '/home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/dpo/chat_template.jinja',
 '/home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/dpo/tokenizer.json')

In [11]:
def generate_response(model, prompt, max_new_tokens=96):
    model.eval()
    device = next(model.parameters()).device
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_ids = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


idx = 0
row = dev_rows[idx]

print("Prompt:\n", row["prompt"])
print("\nChosen / Gold:\n", row["chosen"])
print("\nRejected:\n", row["rejected"])
print("\nAfter DPO / Policy Generation:\n", generate_response(policy_model, row["prompt"]))


Prompt:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
A teammate insists on frequent communication to feel secure.

Question:
How would you handle the teammate’s demand for constant updates?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:


Chosen / Gold:
 I would prioritize maintaining that frequent contact despite my schedule, as I want to ensure my teammate feels valued, included, and that our teamwork remains perfectly smooth.

Rejected:
 I would suggest we set strict boundaries to focus on efficiency, telling them that my autonomy and time management are more important than their need for constant updates.

After DPO / Policy Generation:
 I would make sure to provide regular, respectful updates to maintain harmony and make them feel included, avo